# Agent 3 — Environment Agent

Domain agent 3 of 8 in ImmunoSense. Captures invisible environmental triggers (PM2.5, ozone, UV, barometric pressure, pollen) that affect autoimmune disease but which patients cannot observe themselves.

**Three-layer architecture** (consistent with Agents 1, 2, 4):
- **Layer 1**: Geographic regional baseline (hardcoded EPA/NOAA reference tables)
- **Layer 2**: API ingestion (AirNow + OpenWeather + Open-Meteo + Google Pollen) → daily 5-feature summary
- **Layer 3**: Per-patient adaptation (robust tracker + BH FDR-corrected trigger detector)

**Output contract:** `EnvironmentAgent.observe(daily_summary)`, `observe_flare(date, severity)`, `analyze() → EnvironmentAgentReport`, `flare_signature() → conductor-facing 0-1 score`. Same shape as `DietaryAgent` for uniform Conductor consumption.

## Known limitations (handled by Conductor or accepted for v1)

1. **Sparse-trigger sensitivity**: BH FDR at α=0.10 suppresses weak signals. Triggers firing <15% of days may not surface in 60-day windows. Real deployment with multi-month observation should improve detection.
2. **Regional norms are approximations**: Layer 1 reference values transcribed from training-data knowledge. Verify against current EPA AQS and NOAA publications before clinical use.
3. **Day 1 barometric is synthetic**: `barometric_change_kpa` requires yesterday's cached pressure for the 24h delta. First call falls back to Mock; real values flow from day 2 onward.
4. **Pollen aggregation is max-based**: Tree/grass/weed indices collapsed via max(). Patients with specific allergen sensitivity (e.g., grass only) would benefit from per-category tracking in v2.
5. **Cross-agent corroboration is the Conductor's job**: Agent 3 produces FDR-controlled findings within its own hypothesis space. Conductor (Agent 9) must corroborate single-agent signals against other agents before acting clinically.

In [ ]:
# ============================================================================
# Section 1: Setup
# ============================================================================
import os
import pickle
import hashlib
import random as random_mod
from collections import deque
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Optional, Protocol

import numpy as np
import pandas as pd
import requests
from scipy.stats import norm

from dotenv import load_dotenv

ENV_PATH = Path.cwd().parent / '.env' if Path.cwd().name == 'notebooks' else Path.cwd() / '.env'
load_dotenv(ENV_PATH)

def _key_status(name: str) -> str:
    val = os.environ.get(name)
    return f'OK ({len(val)} chars)' if val else 'NOT SET - Mock fallback'

print('API key status:')
print(f'  AIRNOW_API_KEY:        {_key_status("AIRNOW_API_KEY")}')
print(f'  OPENWEATHER_API_KEY:   {_key_status("OPENWEATHER_API_KEY")}')
print(f'  GOOGLE_POLLEN_API_KEY: {_key_status("GOOGLE_POLLEN_API_KEY")}')

ARTIFACT_DIR = Path('./artifacts/agent3')
CACHE_DIR = ARTIFACT_DIR / 'api_cache'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f'\nArtifact dir: {ARTIFACT_DIR.resolve()}')
print(f'.env exists: {ENV_PATH.exists()}')

In [ ]:
# ============================================================================
# Section 2: Layer 1 - Regional Baseline Tables
#
# 5 US regions x 4 seasons x 5 features = 100 (mean, std) cells
# Sources (approximations - verify before clinical use):
#   PM2.5/ozone: EPA AQS 2018-2023 averages
#   UV:          WHO Global Solar UV Index regional means
#   Barometric:  NOAA climatology, |24h-change| magnitudes
#   Pollen:      NAB station data, Universal Pollen Index 0-5
# ============================================================================
REGIONS = ['NE', 'SE', 'MW', 'W', 'SW']
SEASONS = ['spring', 'summer', 'fall', 'winter']
FEATURES = ['pm25', 'ozone', 'uv', 'barometric', 'pollen']

REGIONAL_NORMS = {
    'NE': {
        'spring': {'pm25': (8.5, 3.5),  'ozone': (45, 10), 'uv': (5.5, 1.5),  'barometric': (0.4, 0.3), 'pollen': (5.5, 2.0)},
        'summer': {'pm25': (9.0, 4.0),  'ozone': (55, 12), 'uv': (8.0, 1.5),  'barometric': (0.3, 0.2), 'pollen': (3.5, 1.5)},
        'fall':   {'pm25': (8.0, 3.5),  'ozone': (40, 9),  'uv': (4.0, 1.5),  'barometric': (0.5, 0.3), 'pollen': (4.5, 2.0)},
        'winter': {'pm25': (9.5, 4.0),  'ozone': (32, 7),  'uv': (2.0, 1.0),  'barometric': (0.6, 0.4), 'pollen': (1.0, 0.8)},
    },
    'SE': {
        'spring': {'pm25': (9.0, 3.5),  'ozone': (50, 11), 'uv': (7.0, 1.5),  'barometric': (0.3, 0.3), 'pollen': (7.5, 2.0)},
        'summer': {'pm25': (10.0, 4.5), 'ozone': (60, 13), 'uv': (9.5, 1.5),  'barometric': (0.3, 0.3), 'pollen': (4.0, 1.5)},
        'fall':   {'pm25': (8.5, 3.5),  'ozone': (42, 10), 'uv': (5.5, 1.5),  'barometric': (0.4, 0.3), 'pollen': (5.5, 2.0)},
        'winter': {'pm25': (9.0, 4.0),  'ozone': (35, 8),  'uv': (3.0, 1.0),  'barometric': (0.5, 0.3), 'pollen': (2.0, 1.0)},
    },
    'MW': {
        'spring': {'pm25': (9.5, 4.0),  'ozone': (45, 10), 'uv': (5.5, 1.5),  'barometric': (0.5, 0.4), 'pollen': (6.0, 2.0)},
        'summer': {'pm25': (10.5, 4.5), 'ozone': (58, 12), 'uv': (8.5, 1.5),  'barometric': (0.4, 0.3), 'pollen': (4.5, 1.5)},
        'fall':   {'pm25': (9.0, 4.0),  'ozone': (40, 9),  'uv': (4.0, 1.5),  'barometric': (0.5, 0.4), 'pollen': (5.0, 2.0)},
        'winter': {'pm25': (11.0, 5.0), 'ozone': (30, 7),  'uv': (1.5, 0.8),  'barometric': (0.7, 0.5), 'pollen': (1.0, 0.8)},
    },
    'W': {
        'spring': {'pm25': (8.0, 4.0),  'ozone': (50, 11), 'uv': (6.5, 1.5),  'barometric': (0.4, 0.3), 'pollen': (5.0, 2.0)},
        'summer': {'pm25': (12.0, 7.0), 'ozone': (62, 14), 'uv': (9.0, 1.5),  'barometric': (0.3, 0.2), 'pollen': (3.0, 1.5)},
        'fall':   {'pm25': (11.0, 6.5), 'ozone': (45, 10), 'uv': (4.5, 1.5),  'barometric': (0.4, 0.3), 'pollen': (4.0, 2.0)},
        'winter': {'pm25': (9.5, 5.0),  'ozone': (28, 7),  'uv': (2.5, 1.0),  'barometric': (0.6, 0.4), 'pollen': (1.5, 1.0)},
    },
    'SW': {
        'spring': {'pm25': (8.5, 3.5),  'ozone': (55, 12), 'uv': (7.5, 1.5),  'barometric': (0.4, 0.3), 'pollen': (5.5, 2.0)},
        'summer': {'pm25': (10.0, 4.0), 'ozone': (68, 14), 'uv': (10.5, 1.5), 'barometric': (0.3, 0.3), 'pollen': (3.5, 1.5)},
        'fall':   {'pm25': (8.5, 3.5),  'ozone': (48, 11), 'uv': (5.5, 1.5),  'barometric': (0.4, 0.3), 'pollen': (4.5, 2.0)},
        'winter': {'pm25': (8.0, 3.5),  'ozone': (38, 9),  'uv': (3.0, 1.0),  'barometric': (0.5, 0.3), 'pollen': (1.5, 1.0)},
    },
}

# Sanity check
missing = [(r, s, f) for r in REGIONS for s in SEASONS for f in FEATURES
           if f not in REGIONAL_NORMS[r][s]]
assert not missing, f'Missing cells: {missing}'
print(f'Regional norms: {len(REGIONS)} regions x {len(SEASONS)} seasons x {len(FEATURES)} features = {len(REGIONS)*len(SEASONS)*len(FEATURES)} cells, all populated.')

In [ ]:
# ============================================================================
# Section 3: Layer 1 - Percentile Lookup API
#
# Public API consumed by Layer 2:
#   infer_region_from_zip(zip_code) -> 'NE'|'SE'|'MW'|'W'|'SW'
#   infer_season_from_date(date)    -> 'spring'|'summer'|'fall'|'winter'
#   get_population_percentile(region, season, feature, value) -> float [0,1]
# ============================================================================
ZIP_FIRST_DIGIT_TO_REGION = {
    '0': 'NE', '1': 'NE', '2': 'SE', '3': 'SE',
    '4': 'MW', '5': 'MW', '6': 'MW',
    '7': 'SW', '8': 'SW', '9': 'W',
}

def infer_region_from_zip(zip_code) -> str:
    if zip_code is None:
        return 'SE'
    zip_str = str(zip_code).strip().zfill(5)[:5]
    if not zip_str or not zip_str[0].isdigit():
        return 'SE'
    return ZIP_FIRST_DIGIT_TO_REGION.get(zip_str[0], 'SE')

MONTH_TO_SEASON = {
    12: 'winter', 1: 'winter', 2: 'winter',
    3: 'spring', 4: 'spring', 5: 'spring',
    6: 'summer', 7: 'summer', 8: 'summer',
    9: 'fall', 10: 'fall', 11: 'fall',
}

def infer_season_from_date(date) -> str:
    return MONTH_TO_SEASON[pd.Timestamp(date).month]

def get_population_percentile(region: str, season: str, feature: str, value: float) -> float:
    """Place observed value on regional+seasonal population CDF."""
    if region not in REGIONAL_NORMS:
        raise ValueError(f"Unknown region '{region}'")
    if season not in REGIONAL_NORMS[region]:
        raise ValueError(f"Unknown season '{season}'")
    if feature not in REGIONAL_NORMS[region][season]:
        raise ValueError(f"Unknown feature '{feature}'")
    mean, std = REGIONAL_NORMS[region][season][feature]
    if std <= 0:
        return 0.5
    return float(norm.cdf((value - mean) / std))

# Smoke tests
assert infer_region_from_zip('28202') == 'SE'
assert infer_season_from_date('2026-05-24') == 'spring'
assert infer_region_from_zip('85001') == 'SW'
assert infer_region_from_zip('02101') == 'NE'
assert abs(get_population_percentile('SE', 'spring', 'pm25', 9.0) - 0.5) < 0.01
print('Layer 1 functions defined and smoke-tested.')

In [ ]:
# ============================================================================
# Section 4: Layer 2 - Data Source Contracts
#
# FetchedFeatures: raw output from one data source.
# EnvironmentDataSource: Protocol every source implements.
# DailyEnvironmentSummary: Layer 2 output, consumed by Layer 3.
# ============================================================================
@dataclass
class FetchedFeatures:
    pm25_ug_m3: Optional[float] = None
    ozone_ppb: Optional[float] = None
    uv_index: Optional[float] = None
    barometric_change_kpa: Optional[float] = None
    pollen_index: Optional[float] = None
    confidence: dict = field(default_factory=dict)   # feature -> 'real'|'synthetic'
    sources: dict = field(default_factory=dict)      # feature -> source name
    errors: list = field(default_factory=list)


class EnvironmentDataSource(Protocol):
    def fetch(self, latitude: float, longitude: float, target_date: str) -> FetchedFeatures: ...


@dataclass
class DailyEnvironmentSummary:
    """Layer 2 output - one per (patient, date), consumed by Layer 3."""
    date: str
    location: dict
    pm25_ug_m3: Optional[float]
    ozone_ppb: Optional[float]
    uv_index: Optional[float]
    barometric_change_kpa: Optional[float]
    pollen_index: Optional[float]
    percentiles: dict = field(default_factory=dict)
    threshold_alerts: dict = field(default_factory=dict)
    feature_confidence: dict = field(default_factory=dict)
    sources: dict = field(default_factory=dict)
    errors: list = field(default_factory=list)
    overall_confidence: float = 0.0   # fraction of features that are 'real'

print('Data source contracts defined.')

In [ ]:
# ============================================================================
# Section 5: EPA/WHO Threshold Classifiers + AQI Converter
#
# Deterministic clinical thresholds applied per-feature.
# AirNow returns AQI (0-500); we invert EPA's breakpoint table to ug/m^3 / ppb.
# ============================================================================
def classify_pm25_threshold(pm25_ug_m3: Optional[float]) -> Optional[str]:
    """EPA NAAQS PM2.5 24-hour AQI breakpoints."""
    if pm25_ug_m3 is None:
        return None
    if pm25_ug_m3 <= 12.0:  return 'good'
    if pm25_ug_m3 <= 35.4:  return 'moderate'
    if pm25_ug_m3 <= 55.4:  return 'unhealthy_sensitive'   # autoimmune alert
    if pm25_ug_m3 <= 150.4: return 'unhealthy'
    if pm25_ug_m3 <= 250.4: return 'very_unhealthy'
    return 'hazardous'


def classify_ozone_threshold(ozone_ppb: Optional[float]) -> Optional[str]:
    """EPA NAAQS 8-hour ozone AQI breakpoints."""
    if ozone_ppb is None:
        return None
    if ozone_ppb <= 54:  return 'good'
    if ozone_ppb <= 70:  return 'moderate'
    if ozone_ppb <= 85:  return 'unhealthy_sensitive'
    if ozone_ppb <= 105: return 'unhealthy'
    if ozone_ppb <= 200: return 'very_unhealthy'
    return 'hazardous'


def classify_uv_threshold(uv_index: Optional[float]) -> Optional[str]:
    """WHO Global Solar UV Index. UV >= 6 = SLE photosensitivity alert."""
    if uv_index is None:
        return None
    if uv_index < 3:  return 'low'
    if uv_index < 6:  return 'moderate'
    if uv_index < 8:  return 'high'           # autoimmune alert
    if uv_index < 11: return 'very_high'
    return 'extreme'


def classify_barometric_threshold(barometric_change_kpa: Optional[float]) -> Optional[str]:
    """24h pressure change magnitude. >= 0.8 kPa = RA joint pain risk (Smedslund 2010)."""
    if barometric_change_kpa is None:
        return None
    abs_change = abs(barometric_change_kpa)
    if abs_change < 0.3: return 'stable'
    if abs_change < 0.8: return 'moderate'
    if abs_change < 1.5: return 'large'       # RA joint pain alert
    return 'extreme'


def classify_pollen_threshold(pollen_index: Optional[float]) -> Optional[str]:
    """Google Universal Pollen Index (0-5)."""
    if pollen_index is None:
        return None
    if pollen_index < 1: return 'very_low'
    if pollen_index < 2: return 'low'
    if pollen_index < 3: return 'moderate'
    if pollen_index < 4: return 'high'
    return 'very_high'


def classify_all_thresholds(features: FetchedFeatures) -> dict:
    return {
        'pm25':       classify_pm25_threshold(features.pm25_ug_m3),
        'ozone':      classify_ozone_threshold(features.ozone_ppb),
        'uv':         classify_uv_threshold(features.uv_index),
        'barometric': classify_barometric_threshold(features.barometric_change_kpa),
        'pollen':     classify_pollen_threshold(features.pollen_index),
    }


# EPA AQI breakpoint inversion: AQI -> concentration via linear interpolation
PM25_AQI_BREAKPOINTS = [
    (0, 50, 0.0, 12.0), (51, 100, 12.1, 35.4), (101, 150, 35.5, 55.4),
    (151, 200, 55.5, 150.4), (201, 300, 150.5, 250.4),
    (301, 400, 250.5, 350.4), (401, 500, 350.5, 500.4),
]
OZONE_AQI_BREAKPOINTS_8HR = [
    (0, 50, 0, 54), (51, 100, 55, 70), (101, 150, 71, 85),
    (151, 200, 86, 105), (201, 300, 106, 200),
]

def aqi_to_concentration(aqi, breakpoints) -> Optional[float]:
    if aqi is None:
        return None
    aqi = float(aqi)
    for (aqi_lo, aqi_hi, conc_lo, conc_hi) in breakpoints:
        if aqi_lo <= aqi <= aqi_hi:
            frac = (aqi - aqi_lo) / (aqi_hi - aqi_lo) if aqi_hi > aqi_lo else 0.0
            return conc_lo + frac * (conc_hi - conc_lo)
    return None

def aqi_to_pm25(aqi):       return aqi_to_concentration(aqi, PM25_AQI_BREAKPOINTS)
def aqi_to_ozone_ppb(aqi):  return aqi_to_concentration(aqi, OZONE_AQI_BREAKPOINTS_8HR)

print('Threshold classifiers + AQI converter defined.')

In [ ]:
# ============================================================================
# Section 6: MockEnvironmentSource
#
# Synthetic data sampled from REGIONAL_NORMS. Deterministic per (lat, lon, date)
# so notebook reruns are stable. Used as fallback when real APIs unavailable.
# ============================================================================
class MockEnvironmentSource:
    def __init__(self, seed_offset: int = 0):
        self.seed_offset = seed_offset

    def fetch(self, latitude: float, longitude: float, target_date: str) -> FetchedFeatures:
        seed_key = f'{latitude:.4f},{longitude:.4f},{target_date},{self.seed_offset}'
        seed = int(hashlib.md5(seed_key.encode()).hexdigest()[:8], 16)
        rng = random_mod.Random(seed)

        region = self._region_from_latlon(latitude, longitude)
        season = infer_season_from_date(target_date)
        norms = REGIONAL_NORMS[region][season]

        pm25 = max(0.0, rng.gauss(norms['pm25'][0], norms['pm25'][1]))
        ozone = max(0.0, rng.gauss(norms['ozone'][0], norms['ozone'][1]))
        uv = max(0.0, min(12.0, rng.gauss(norms['uv'][0], norms['uv'][1])))
        barometric = max(0.0, abs(rng.gauss(0.0, norms['barometric'][1])))
        pollen = max(0.0, min(10.0, rng.gauss(norms['pollen'][0], norms['pollen'][1])))

        # 5% chance of spike event (dust storm, pollution event)
        if rng.random() < 0.05:
            spike = rng.choice(['pm25', 'ozone', 'pollen'])
            if spike == 'pm25':   pm25 *= rng.uniform(2.5, 4.0)
            elif spike == 'ozone': ozone *= rng.uniform(1.5, 2.0)
            elif spike == 'pollen': pollen = min(10.0, pollen * rng.uniform(1.8, 2.5))

        return FetchedFeatures(
            pm25_ug_m3=round(pm25, 1),
            ozone_ppb=round(ozone, 0),
            uv_index=round(uv, 1),
            barometric_change_kpa=round(barometric, 2),
            pollen_index=round(pollen, 1),
            confidence={f: 'synthetic' for f in ['pm25_ug_m3', 'ozone_ppb', 'uv_index',
                                                  'barometric_change_kpa', 'pollen_index']},
            sources={f: 'mock' for f in ['pm25_ug_m3', 'ozone_ppb', 'uv_index',
                                          'barometric_change_kpa', 'pollen_index']},
        )

    def _region_from_latlon(self, lat: float, lon: float) -> str:
        if lon > -82:   return 'SE' if lat < 40 else 'NE'
        if lon > -95:   return 'SE' if lat < 36 else 'MW'
        if lon > -105:  return 'SW' if lat < 40 else 'MW'
        return 'W'

print('MockEnvironmentSource defined.')

In [ ]:
# ============================================================================
# Section 7: AirNowSource (real PM2.5 + ozone from EPA AirNow)
# Cached to disk for 24h. Falls back to FetchedFeatures with errors on failure.
# ============================================================================
class AirNowSource:
    BASE_URL = 'https://www.airnowapi.org/aq/observation/latLong/current/'
    DISTANCE_MILES = 25
    CACHE_TTL_HOURS = 24

    def __init__(self, api_key: Optional[str] = None, cache_dir: Optional[Path] = None):
        self.api_key = api_key or os.environ.get('AIRNOW_API_KEY')
        self.cache_dir = cache_dir or CACHE_DIR
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    def fetch(self, latitude: float, longitude: float, target_date: str) -> FetchedFeatures:
        cache_path = self._cache_path(latitude, longitude, target_date)
        if cache_path.exists() and self._cache_is_fresh(cache_path):
            with open(cache_path, 'rb') as f:
                return pickle.load(f)

        if not self.api_key:
            return FetchedFeatures(errors=['AirNow: no AIRNOW_API_KEY'])

        try:
            response = requests.get(self.BASE_URL, params={
                'format': 'application/json',
                'latitude': latitude, 'longitude': longitude,
                'distance': self.DISTANCE_MILES, 'API_KEY': self.api_key,
            }, timeout=10)
            response.raise_for_status()
            data = response.json()
        except requests.exceptions.RequestException as e:
            return FetchedFeatures(errors=[f'AirNow HTTP: {type(e).__name__}: {e}'])
        except ValueError as e:
            return FetchedFeatures(errors=[f'AirNow JSON: {e}'])

        result = self._parse_response(data)
        try:
            with open(cache_path, 'wb') as f:
                pickle.dump(result, f)
        except Exception as e:
            result.errors.append(f'AirNow cache write: {e}')
        return result

    def _parse_response(self, data: list) -> FetchedFeatures:
        if not isinstance(data, list) or not data:
            return FetchedFeatures(errors=['AirNow: empty or invalid response'])

        pm25_ug_m3 = None
        ozone_ppb = None
        for obs in data:
            param = obs.get('ParameterName', '').upper()
            aqi = obs.get('AQI')
            if param == 'PM2.5' and aqi is not None:
                pm25_ug_m3 = aqi_to_pm25(aqi)
            elif param in ('O3', 'OZONE') and aqi is not None:
                ozone_ppb = aqi_to_ozone_ppb(aqi)

        confidence, sources = {}, {}
        if pm25_ug_m3 is not None:
            confidence['pm25_ug_m3'] = 'real'
            sources['pm25_ug_m3'] = 'airnow'
        if ozone_ppb is not None:
            confidence['ozone_ppb'] = 'real'
            sources['ozone_ppb'] = 'airnow'

        return FetchedFeatures(
            pm25_ug_m3=pm25_ug_m3, ozone_ppb=ozone_ppb,
            confidence=confidence, sources=sources,
        )

    def _cache_path(self, lat: float, lon: float, target_date: str) -> Path:
        return self.cache_dir / f'airnow_{lat:.4f}_{lon:.4f}_{target_date}.pkl'

    def _cache_is_fresh(self, cache_path: Path) -> bool:
        age_hours = (datetime.now().timestamp() - cache_path.stat().st_mtime) / 3600
        return age_hours < self.CACHE_TTL_HOURS

print('AirNowSource defined.')

In [ ]:
# ============================================================================
# Section 8: OpenWeatherSource (barometric pressure + UV)
#
# Uses TWO upstream APIs since OpenWeather free tier doesn't include UV:
#   - OpenWeather /weather       -> pressure (main.pressure, hPa)
#   - Open-Meteo /forecast       -> UV index (daily.uv_index_max, no API key)
#
# Barometric semantics:
#   Day 1 for a location: pressure stored to disk, change=None
#   Day 2+: change = today's pressure - yesterday's stored pressure
# ============================================================================
class OpenWeatherSource:
    OPENWEATHER_URL = 'https://api.openweathermap.org/data/2.5/weather'
    OPEN_METEO_URL = 'https://api.open-meteo.com/v1/forecast'
    CACHE_TTL_HOURS = 6

    def __init__(self, api_key: Optional[str] = None, cache_dir: Optional[Path] = None):
        self.api_key = api_key or os.environ.get('OPENWEATHER_API_KEY')
        self.cache_dir = cache_dir or CACHE_DIR
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    def fetch(self, latitude: float, longitude: float, target_date: str) -> FetchedFeatures:
        cache_path = self._cache_path(latitude, longitude, target_date)
        if cache_path.exists() and self._cache_is_fresh(cache_path):
            with open(cache_path, 'rb') as f:
                return pickle.load(f)

        pressure_kpa, ow_errors = self._fetch_pressure(latitude, longitude)
        uv_index, om_errors = self._fetch_uv(latitude, longitude)

        # Store today's raw pressure to a sidecar file for tomorrow's delta
        if pressure_kpa is not None:
            self._save_raw_pressure(latitude, longitude, target_date, pressure_kpa)

        barometric_change_kpa = self._compute_pressure_delta(
            latitude, longitude, target_date, pressure_kpa
        )

        confidence, sources = {}, {}
        if barometric_change_kpa is not None:
            confidence['barometric_change_kpa'] = 'real'
            sources['barometric_change_kpa'] = 'openweather'
        if uv_index is not None:
            confidence['uv_index'] = 'real'
            sources['uv_index'] = 'open-meteo'

        result = FetchedFeatures(
            barometric_change_kpa=round(barometric_change_kpa, 2) if barometric_change_kpa is not None else None,
            uv_index=round(uv_index, 1) if uv_index is not None else None,
            confidence=confidence, sources=sources,
            errors=ow_errors + om_errors,
        )

        try:
            with open(cache_path, 'wb') as f:
                pickle.dump(result, f)
        except Exception as e:
            result.errors.append(f'OpenWeather cache write: {e}')
        return result

    def _fetch_pressure(self, latitude, longitude):
        if not self.api_key:
            return None, ['OpenWeather: no OPENWEATHER_API_KEY']
        try:
            response = requests.get(self.OPENWEATHER_URL, params={
                'lat': latitude, 'lon': longitude,
                'appid': self.api_key, 'units': 'metric',
            }, timeout=10)
            data = response.json()
        except requests.exceptions.RequestException as e:
            return None, [f'OpenWeather HTTP: {type(e).__name__}: {e}']
        except ValueError as e:
            return None, [f'OpenWeather JSON: {e}']

        if 'cod' in data and str(data.get('cod')) != '200':
            return None, [f'OpenWeather API: {data.get("message", "unknown")}']

        pressure_hpa = data.get('main', {}).get('pressure')
        if pressure_hpa is None:
            return None, ['OpenWeather: main.pressure missing']
        return pressure_hpa / 10.0, []

    def _fetch_uv(self, latitude, longitude):
        try:
            response = requests.get(self.OPEN_METEO_URL, params={
                'latitude': latitude, 'longitude': longitude,
                'daily': 'uv_index_max', 'forecast_days': 1, 'timezone': 'auto',
            }, timeout=10)
            data = response.json()
        except requests.exceptions.RequestException as e:
            return None, [f'Open-Meteo HTTP: {type(e).__name__}: {e}']
        except ValueError as e:
            return None, [f'Open-Meteo JSON: {e}']

        try:
            uv_max = data['daily']['uv_index_max'][0]
            if uv_max is None:
                return None, ['Open-Meteo: uv_index_max null']
            return float(uv_max), []
        except (KeyError, IndexError, TypeError) as e:
            return None, [f'Open-Meteo unexpected shape: {e}']

    def _save_raw_pressure(self, lat, lon, target_date, pressure_kpa):
        path = self._pressure_path(lat, lon, target_date)
        try:
            with open(path, 'wb') as f:
                pickle.dump(pressure_kpa, f)
        except Exception:
            pass   # non-critical

    def _compute_pressure_delta(self, lat, lon, target_date, current_pressure_kpa):
        if current_pressure_kpa is None:
            return None
        yesterday = (pd.Timestamp(target_date) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        yesterday_path = self._pressure_path(lat, lon, yesterday)
        if not yesterday_path.exists():
            return None   # first day for this location
        try:
            with open(yesterday_path, 'rb') as f:
                prior = pickle.load(f)
            if not isinstance(prior, (int, float)):
                return None
            return current_pressure_kpa - prior
        except Exception:
            return None

    def _cache_path(self, lat, lon, target_date):
        return self.cache_dir / f'openweather_{lat:.4f}_{lon:.4f}_{target_date}.pkl'

    def _pressure_path(self, lat, lon, target_date):
        return self.cache_dir / f'pressure_{lat:.4f}_{lon:.4f}_{target_date}.pkl'

    def _cache_is_fresh(self, cache_path):
        age_hours = (datetime.now().timestamp() - cache_path.stat().st_mtime) / 3600
        return age_hours < self.CACHE_TTL_HOURS

print('OpenWeatherSource defined (OpenWeather pressure + Open-Meteo UV).')

In [ ]:
# ============================================================================
# Section 9: GooglePollenSource (tree + grass + weed pollen, max-aggregated)
# ============================================================================
class GooglePollenSource:
    BASE_URL = 'https://pollen.googleapis.com/v1/forecast:lookup'
    CACHE_TTL_HOURS = 24

    def __init__(self, api_key: Optional[str] = None, cache_dir: Optional[Path] = None):
        self.api_key = api_key or os.environ.get('GOOGLE_POLLEN_API_KEY')
        self.cache_dir = cache_dir or CACHE_DIR
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    def fetch(self, latitude: float, longitude: float, target_date: str) -> FetchedFeatures:
        cache_path = self._cache_path(latitude, longitude, target_date)
        if cache_path.exists() and self._cache_is_fresh(cache_path):
            with open(cache_path, 'rb') as f:
                return pickle.load(f)

        if not self.api_key:
            return FetchedFeatures(errors=['GooglePollen: no GOOGLE_POLLEN_API_KEY'])

        try:
            response = requests.get(self.BASE_URL, params={
                'key': self.api_key,
                'location.latitude': latitude, 'location.longitude': longitude,
                'days': 1,
            }, timeout=10)
            data = response.json()
        except requests.exceptions.RequestException as e:
            return FetchedFeatures(errors=[f'GooglePollen HTTP: {type(e).__name__}: {e}'])
        except ValueError as e:
            return FetchedFeatures(errors=[f'GooglePollen JSON: {e}'])

        if 'error' in data:
            err = data['error']
            return FetchedFeatures(errors=[f'GooglePollen API {err.get("code")}: {err.get("message")}'])

        result = self._parse_response(data)
        try:
            with open(cache_path, 'wb') as f:
                pickle.dump(result, f)
        except Exception as e:
            result.errors.append(f'GooglePollen cache write: {e}')
        return result

    def _parse_response(self, data: dict) -> FetchedFeatures:
        try:
            daily_info = data.get('dailyInfo', [])
            if not daily_info:
                return FetchedFeatures(errors=['GooglePollen: dailyInfo missing'])

            today = daily_info[0]
            indices = {}
            for entry in today.get('pollenTypeInfo', []):
                code = entry.get('code', '').upper()
                if code in ('TREE', 'GRASS', 'WEED'):
                    value = entry.get('indexInfo', {}).get('value')
                    if value is not None:
                        indices[code] = value

            if not indices:
                return FetchedFeatures(errors=['GooglePollen: no indices in response'])

            # Max aggregation: patient feels the worst trigger, not the average
            pollen_max = float(max(indices.values()))
            return FetchedFeatures(
                pollen_index=round(pollen_max, 1),
                confidence={'pollen_index': 'real'},
                sources={'pollen_index': 'google-pollen'},
            )
        except (KeyError, IndexError, TypeError) as e:
            return FetchedFeatures(errors=[f'GooglePollen unexpected shape: {type(e).__name__}: {e}'])

    def _cache_path(self, lat, lon, target_date):
        return self.cache_dir / f'pollen_{lat:.4f}_{lon:.4f}_{target_date}.pkl'

    def _cache_is_fresh(self, cache_path):
        age_hours = (datetime.now().timestamp() - cache_path.stat().st_mtime) / 3600
        return age_hours < self.CACHE_TTL_HOURS

print('GooglePollenSource defined.')

In [ ]:
# ============================================================================
# Section 10: CompositeEnvironmentSource (production data source)
#
# Wraps all three real APIs + Mock fallback. Per-feature graceful degradation:
# if a real source returns None for a feature, Mock fills the gap and the
# confidence is stamped 'synthetic'.
# ============================================================================
class CompositeEnvironmentSource:
    AIRNOW_FEATURES   = ['pm25_ug_m3', 'ozone_ppb']
    WEATHER_FEATURES  = ['barometric_change_kpa', 'uv_index']
    POLLEN_FEATURES   = ['pollen_index']

    def __init__(self,
                 airnow: Optional[AirNowSource] = None,
                 openweather: Optional[OpenWeatherSource] = None,
                 google_pollen: Optional[GooglePollenSource] = None,
                 mock: Optional[MockEnvironmentSource] = None):
        self.airnow = airnow or AirNowSource()
        self.openweather = openweather or OpenWeatherSource()
        self.google_pollen = google_pollen or GooglePollenSource()
        self.mock = mock or MockEnvironmentSource()

    def fetch(self, latitude: float, longitude: float, target_date: str) -> FetchedFeatures:
        airnow_result   = self.airnow.fetch(latitude, longitude, target_date)
        weather_result  = self.openweather.fetch(latitude, longitude, target_date)
        pollen_result   = self.google_pollen.fetch(latitude, longitude, target_date)
        mock_result     = self.mock.fetch(latitude, longitude, target_date)

        merged = FetchedFeatures()

        for feature in self.AIRNOW_FEATURES:
            self._merge_feature(merged, feature, airnow_result, mock_result, 'airnow')

        for feature in self.WEATHER_FEATURES:
            real_source = weather_result.sources.get(feature, 'openweather')
            self._merge_feature(merged, feature, weather_result, mock_result, real_source)

        for feature in self.POLLEN_FEATURES:
            self._merge_feature(merged, feature, pollen_result, mock_result, 'google-pollen')

        merged.errors = (airnow_result.errors + weather_result.errors + pollen_result.errors)
        return merged

    def _merge_feature(self, merged, feature, real_result, mock_result, real_source_name):
        real_val = getattr(real_result, feature)
        if real_val is not None:
            setattr(merged, feature, real_val)
            merged.confidence[feature] = 'real'
            merged.sources[feature] = real_source_name
        else:
            setattr(merged, feature, getattr(mock_result, feature))
            merged.confidence[feature] = 'synthetic'
            merged.sources[feature] = 'mock'

print('CompositeEnvironmentSource defined.')

In [ ]:
# ============================================================================
# Section 11: Layer 2 Pipeline - process_environment_day
#
# Pipeline: ZIP + date + source -> DailyEnvironmentSummary
#   1. ZIP -> coordinates and region/season
#   2. Fetch raw features from data source
#   3. Compute Layer 1 percentiles per feature
#   4. Apply EPA/WHO threshold classifiers
#   5. Compute overall_confidence (fraction of features that are 'real')
# ============================================================================
ZIP_APPROX_COORDS = {
    '28202': (35.227, -80.843),    # Charlotte, NC
    '85001': (33.448, -112.074),   # Phoenix, AZ
    '02101': (42.358, -71.060),    # Boston, MA
    '60601': (41.886, -87.625),    # Chicago, IL
    '90001': (33.974, -118.249),   # Los Angeles, CA
    '10001': (40.751, -74.000),    # New York, NY
    '94102': (37.779, -122.419),   # San Francisco, CA
    '77001': (29.749, -95.358),    # Houston, TX
    '33101': (25.789, -80.224),    # Miami, FL
    '80201': (39.741, -104.984),   # Denver, CO
}

REGION_CENTERS = {
    'NE': (42.0, -75.0),  'SE': (35.0, -82.0),  'MW': (40.0, -88.0),
    'W':  (38.0, -120.0), 'SW': (34.0, -106.0),
}

def zip_to_latlon(zip_code) -> tuple:
    zip_str = str(zip_code).zfill(5)[:5]
    if zip_str in ZIP_APPROX_COORDS:
        return ZIP_APPROX_COORDS[zip_str]
    return REGION_CENTERS.get(infer_region_from_zip(zip_str), (35.0, -82.0))


# Map FetchedFeatures field names to Layer 1 feature names
_FF_TO_L1 = {
    'pm25_ug_m3': 'pm25',
    'ozone_ppb': 'ozone',
    'uv_index': 'uv',
    'barometric_change_kpa': 'barometric',
    'pollen_index': 'pollen',
}


def process_environment_day(
    zip_code: str,
    target_date: str,
    source: Optional[EnvironmentDataSource] = None,
) -> DailyEnvironmentSummary:
    """Layer 2 pipeline. ZIP + date -> structured summary for Layer 3."""
    if source is None:
        source = CompositeEnvironmentSource()

    lat, lon = zip_to_latlon(zip_code)
    region = infer_region_from_zip(zip_code)
    season = infer_season_from_date(target_date)

    fetched = source.fetch(lat, lon, target_date)

    percentiles = {}
    for ff_field, l1_name in _FF_TO_L1.items():
        value = getattr(fetched, ff_field)
        percentiles[l1_name] = (
            None if value is None
            else get_population_percentile(region, season, l1_name, value)
        )

    threshold_alerts = classify_all_thresholds(fetched)

    n_real = sum(1 for v in fetched.confidence.values() if v == 'real')
    overall_confidence = n_real / 5.0 if fetched.confidence else 0.0

    return DailyEnvironmentSummary(
        date=target_date,
        location={'zip': str(zip_code).zfill(5)[:5], 'lat': lat, 'lon': lon,
                  'region': region, 'season': season},
        pm25_ug_m3=fetched.pm25_ug_m3,
        ozone_ppb=fetched.ozone_ppb,
        uv_index=fetched.uv_index,
        barometric_change_kpa=fetched.barometric_change_kpa,
        pollen_index=fetched.pollen_index,
        percentiles=percentiles,
        threshold_alerts=threshold_alerts,
        feature_confidence=dict(fetched.confidence),
        sources=dict(fetched.sources),
        errors=list(fetched.errors),
        overall_confidence=overall_confidence,
    )

print('Layer 2 pipeline (process_environment_day) defined.')

In [ ]:
# ============================================================================
# Section 12: Layer 3 - Robust Baseline Tracker
#
# Five parallel univariate trackers, one per environmental feature.
# Rolling median + IQR with outlier exclusion. Same pattern as Agent 2 Layer 3.
# Activates at 3 days of clean history; full personalization weight by ~25 days.
# ============================================================================
ENV_FEATURES = [
    'pm25_ug_m3', 'ozone_ppb', 'uv_index',
    'barometric_change_kpa', 'pollen_index',
]


class _UnivariateRobustTracker:
    """Rolling median + IQR with outlier exclusion for one feature."""

    def __init__(self, window=14, anomaly_threshold=2.0, personalization_days=25):
        self.window = window
        self.anomaly_threshold = anomaly_threshold
        self.personalization_days = personalization_days
        self.all_history = deque(maxlen=window)
        self.clean_history = deque(maxlen=window)

    def update(self, value):
        if value is None or pd.isna(value):
            return
        self.all_history.append(float(value))

        if len(self.clean_history) < 3:
            self.clean_history.append(float(value))
        else:
            arr = np.array(self.clean_history)
            med = np.median(arr)
            q75, q25 = np.percentile(arr, [75, 25])
            iqr = q75 - q25
            if iqr < 1e-9:
                self.clean_history.append(float(value))
            elif abs(value - med) / iqr < self.anomaly_threshold:
                self.clean_history.append(float(value))

    def baseline(self):
        n_clean = len(self.clean_history)
        if n_clean < 3:
            return {'median': float('nan'), 'iqr': float('nan'),
                    'n_clean': n_clean, 'personal_weight': 0.0}
        arr = np.array(self.clean_history)
        med = float(np.median(arr))
        q75, q25 = np.percentile(arr, [75, 25])
        return {
            'median': med,
            'iqr': float(q75 - q25),
            'n_clean': n_clean,
            'personal_weight': min(1.0, n_clean / self.personalization_days),
        }

    def anomaly_score(self, value):
        if pd.isna(value) or len(self.clean_history) < 3:
            return float('nan')
        arr = np.array(self.clean_history)
        med = np.median(arr)
        q75, q25 = np.percentile(arr, [75, 25])
        iqr = q75 - q25
        if iqr < 1e-9:
            return 0.0
        return float((value - med) / iqr)


class EnvironmentRobustTracker:
    """Per-patient baseline for the 5 environmental features."""

    def __init__(self, window=14, anomaly_threshold=2.0, personalization_days=25):
        self.trackers = {
            f: _UnivariateRobustTracker(window, anomaly_threshold, personalization_days)
            for f in ENV_FEATURES
        }

    def update(self, daily_summary):
        for f in ENV_FEATURES:
            v = getattr(daily_summary, f, None)
            if v is not None:
                self.trackers[f].update(v)

    def report(self):
        return {f: t.baseline() for f, t in self.trackers.items()}

    def anomaly_scores(self, daily_summary):
        return {f: self.trackers[f].anomaly_score(getattr(daily_summary, f, float('nan')))
                for f in ENV_FEATURES}

print('EnvironmentRobustTracker defined.')

In [ ]:
# ============================================================================
# Section 13: Layer 3 - Trigger Detector (BH FDR-corrected)
#
# Single threshold pathway (p85), one-sided permutation tests, BH at FDR=0.10.
# Hypothesis count per patient: 5 features x 4 lags x 1 pathway = 20.
#
# Confidence tiers from BH-adjusted q-values (not raw p):
#   q < 0.01 -> 'high'
#   q < 0.05 -> 'medium'
#   q < 0.10 -> 'low'
#
# Findings with q >= 0.10 are suppressed (FDR target).
#
# Design choices:
# - Linear (Pearson) pathway dropped: real autoimmune triggers are threshold-shaped.
# - Only p85 percentile (not p75/p90): captures moderate elevation without
#   being too sensitive to noise (p75) or too strict for moderate triggers (p90).
# - Wrong-direction findings (high values -> fewer flares) are included with
#   p=1.0 in the BH testing surface, NOT dropped. Dropping them before BH
#   would break the multiple-testing correction.
# ============================================================================
@dataclass
class DetectedPattern:
    feature: str
    lag_days: int
    effect_size: float
    p_value: float        # raw permutation p-value
    q_value: float        # BH-adjusted q-value
    n_observations: int
    confidence: str       # 'high' | 'medium' | 'low'


class EnvironmentTriggerDetector:
    def __init__(self, min_days=14, lags=(0, 1, 2, 3),
                 n_permutations=500, binarize_percentile=85,
                 fdr_target=0.10, random_seed=42):
        self.min_days = min_days
        self.lags = lags
        self.n_permutations = n_permutations
        self.binarize_percentile = binarize_percentile
        self.fdr_target = fdr_target
        self.rng = np.random.RandomState(random_seed)
        self.last_n_hypotheses = 0

    def detect(self, daily_records, flare_severity_by_date):
        if len(daily_records) < self.min_days:
            self.last_n_hypotheses = 0
            return []

        records_by_date = {r['date']: r for r in daily_records}
        sorted_dates = sorted(records_by_date.keys())
        candidates = []

        for lag in self.lags:
            paired = []
            for i, date in enumerate(sorted_dates):
                target_idx = i + lag
                if target_idx >= len(sorted_dates):
                    continue
                target_date = sorted_dates[target_idx]
                paired.append((date, flare_severity_by_date.get(target_date, 0.0)))

            if len(paired) < self.min_days:
                continue

            flares = np.array([p[1] for p in paired])

            for feat in ENV_FEATURES:
                values = np.array([records_by_date[p[0]].get(feat, np.nan) for p in paired])
                mask = ~np.isnan(values) & ~np.isnan(flares)
                if mask.sum() < self.min_days:
                    continue
                v, f = values[mask], flares[mask]
                if np.std(v) < 1e-9 or np.std(f) < 1e-9:
                    continue

                threshold = float(np.percentile(v, self.binarize_percentile))
                high = v > threshold
                if high.sum() < 2 or high.sum() > len(v) - 2:
                    continue

                diff_obs = float(f[high].mean() - f[~high].mean())

                # Include wrong-direction findings with p=1.0 in BH testing surface
                if diff_obs > 0:
                    p_val = self._perm_p_mean_diff_onesided(high, f, diff_obs)
                else:
                    p_val = 1.0

                candidates.append({
                    'label': f'{feat} (>p{self.binarize_percentile})',
                    'lag': lag, 'effect': diff_obs,
                    'p': p_val, 'n_obs': int(mask.sum()),
                })

        self.last_n_hypotheses = len(candidates)
        if not candidates:
            return []

        # Benjamini-Hochberg FDR correction
        candidates.sort(key=lambda c: c['p'])
        n = len(candidates)
        q_values = [0.0] * n
        running_min = 1.0
        for k in range(n - 1, -1, -1):
            rank = k + 1
            adjusted = candidates[k]['p'] * n / rank
            running_min = min(running_min, adjusted)
            q_values[k] = min(1.0, running_min)

        patterns = []
        for k, c in enumerate(candidates):
            q = q_values[k]
            if q >= self.fdr_target:
                continue
            patterns.append(DetectedPattern(
                feature=c['label'], lag_days=c['lag'],
                effect_size=c['effect'], p_value=c['p'], q_value=q,
                n_observations=c['n_obs'],
                confidence=self._confidence(q),
            ))

        patterns.sort(key=lambda p: abs(p.effect_size), reverse=True)
        return patterns

    def _perm_p_mean_diff_onesided(self, bool_vec, flares, observed_diff):
        perms = np.empty(self.n_permutations)
        for i in range(self.n_permutations):
            shuffled = self.rng.permutation(flares)
            perms[i] = shuffled[bool_vec].mean() - shuffled[~bool_vec].mean()
        return float((perms >= observed_diff).mean())

    def _confidence(self, q):
        if q < 0.01: return 'high'
        if q < 0.05: return 'medium'
        return 'low'

print('EnvironmentTriggerDetector defined (BH FDR at alpha=0.10).')

In [ ]:
# ============================================================================
# Section 14: Conductor-Facing Flare Signature
#
# Daily 0-1 environmental flare-risk score with contributing factors.
# Evidence weights (from design doc literature review):
#   PM2.5      = 0.35 (strongest evidence base)
#   UV         = 0.20 (SLE/dermatomyositis photosensitivity)
#   Ozone      = 0.15
#   Barometric = 0.15 (RA joint pain, Smedslund 2010)
#   Pollen     = 0.15
#
# Per-patient detected triggers boost the effective weight 1.5x.
# ============================================================================
EVIDENCE_WEIGHTS = {
    'pm25_ug_m3':            0.35,
    'uv_index':              0.20,
    'ozone_ppb':             0.15,
    'barometric_change_kpa': 0.15,
    'pollen_index':          0.15,
}


def compute_flare_signature(
    daily_summary: DailyEnvironmentSummary,
    anomaly_scores: dict,
    detected_patterns: Optional[list] = None,
) -> dict:
    """0-1 environmental flare risk + contributing factors."""
    detected_patterns = detected_patterns or []

    triggered_features = set()
    for p in detected_patterns:
        base = p.feature.split(' ')[0]   # 'pm25_ug_m3 (>p85)' -> 'pm25_ug_m3'
        if p.confidence in ('high', 'medium'):
            triggered_features.add(base)

    contributions = []
    total_score = 0.0

    for feature, weight in EVIDENCE_WEIGHTS.items():
        anomaly = anomaly_scores.get(feature, float('nan'))
        if pd.isna(anomaly):
            normalized = 0.0
        else:
            normalized = max(0.0, min(3.0, anomaly)) / 3.0   # clip + normalize

        effective_weight = weight * (1.5 if feature in triggered_features else 1.0)
        contribution = normalized * effective_weight
        total_score += contribution

        contributions.append({
            'feature': feature,
            'anomaly_score': float(anomaly) if not pd.isna(anomaly) else None,
            'normalized': normalized,
            'evidence_weight': weight,
            'effective_weight': effective_weight,
            'is_personal_trigger': feature in triggered_features,
        })

    total_score = min(1.0, total_score)
    contributions.sort(key=lambda c: c['normalized'] * c['effective_weight'], reverse=True)

    return {
        'score': total_score,
        'contributing_factors': contributions,
        'threshold_breaches': [
            f for f, alert in daily_summary.threshold_alerts.items()
            if alert in ('unhealthy_sensitive', 'unhealthy', 'very_unhealthy',
                         'hazardous', 'high', 'very_high', 'extreme', 'large')
        ],
        'data_quality_confidence': daily_summary.overall_confidence,
    }

print('compute_flare_signature defined.')

In [ ]:
# ============================================================================
# Section 15: EnvironmentAgent (Layer 3 orchestrator, Conductor-facing API)
#
# Interface contract (same shape as DietaryAgent for uniform Conductor consumption):
#   observe(daily_summary)         - update Layer 3 trackers, record observation
#   observe_flare(date, severity)  - record flare event for trigger detection
#   analyze()                      - return EnvironmentAgentReport
#   flare_signature(summary=None)  - Conductor-facing 0-1 score for one day
# ============================================================================
@dataclass
class EnvironmentAgentReport:
    n_days_observed: int
    n_flare_events: int
    baselines: dict
    today_anomaly_scores: Optional[dict]
    detected_patterns: list
    tracker_activation: dict
    n_hypotheses_tested: int


class EnvironmentAgent:
    def __init__(self, window=14, anomaly_threshold=2.0, personalization_days=25,
                 trigger_min_days=14, n_permutations=500, fdr_target=0.10):
        self.baseline_tracker = EnvironmentRobustTracker(
            window=window,
            anomaly_threshold=anomaly_threshold,
            personalization_days=personalization_days,
        )
        self.trigger_detector = EnvironmentTriggerDetector(
            min_days=trigger_min_days,
            n_permutations=n_permutations,
            fdr_target=fdr_target,
        )
        self.daily_records = []
        self.flare_events = {}
        self._latest_anomalies = None
        self._latest_summary = None

    def observe(self, daily_summary):
        self.baseline_tracker.update(daily_summary)
        self.daily_records.append({
            'date': daily_summary.date,
            **{f: getattr(daily_summary, f, None) for f in ENV_FEATURES},
        })
        self._latest_anomalies = self.baseline_tracker.anomaly_scores(daily_summary)
        self._latest_summary = daily_summary
        return {'anomaly_scores': self._latest_anomalies}

    def observe_flare(self, date, severity):
        self.flare_events[date] = float(severity)

    def analyze(self):
        flare_dict = dict(self.flare_events)
        for rec in self.daily_records:
            flare_dict.setdefault(rec['date'], 0.0)

        patterns = self.trigger_detector.detect(self.daily_records, flare_dict)
        baselines = self.baseline_tracker.report()
        activation = {f: baselines[f]['n_clean'] >= 3 for f in ENV_FEATURES}
        activation['trigger_detector'] = len(self.daily_records) >= self.trigger_detector.min_days

        return EnvironmentAgentReport(
            n_days_observed=len(self.daily_records),
            n_flare_events=sum(1 for s in self.flare_events.values() if s > 0),
            baselines=baselines,
            today_anomaly_scores=self._latest_anomalies,
            detected_patterns=patterns,
            tracker_activation=activation,
            n_hypotheses_tested=self.trigger_detector.last_n_hypotheses,
        )

    def flare_signature(self, daily_summary=None):
        summary = daily_summary or self._latest_summary
        if summary is None:
            return {
                'score': 0.0, 'contributing_factors': [],
                'threshold_breaches': [], 'data_quality_confidence': 0.0,
            }
        anomalies = self.baseline_tracker.anomaly_scores(summary)
        report = self.analyze()
        return compute_flare_signature(summary, anomalies, report.detected_patterns)

print('EnvironmentAgent defined.')

In [ ]:
# ============================================================================
# Section 16: End-to-end smoke test against live APIs
# Verifies the full Layer 1 + Layer 2 pipeline against real Charlotte data.
# ============================================================================
import datetime as _dt
_today = _dt.date.today().strftime('%Y-%m-%d')

summary = process_environment_day('28202', _today)

print(f'Date:     {summary.date}')
print(f'Location: ZIP {summary.location["zip"]} ({summary.location["region"]} / {summary.location["season"]})')
print(f'          lat={summary.location["lat"]:.3f}, lon={summary.location["lon"]:.3f}')
print()
print('Raw measurements:')
print(f'  pm25_ug_m3:            {summary.pm25_ug_m3}')
print(f'  ozone_ppb:             {summary.ozone_ppb}')
print(f'  uv_index:              {summary.uv_index}')
print(f'  barometric_change_kpa: {summary.barometric_change_kpa}')
print(f'  pollen_index:          {summary.pollen_index}')
print()
print('Population percentiles (vs regional/seasonal norms):')
for feature, pct in summary.percentiles.items():
    pct_str = f'{pct:.0%}' if pct is not None else 'N/A'
    print(f'  {feature:12s}: {pct_str}')
print()
print('EPA/WHO threshold alerts:')
for feature, alert in summary.threshold_alerts.items():
    print(f'  {feature:12s}: {alert}')
print()
print('Feature confidence:')
for feature, conf in summary.feature_confidence.items():
    print(f'  {feature:25s} {conf:10s}  (from {summary.sources.get(feature, "?")})')
print()
print(f'Overall confidence: {summary.overall_confidence:.0%}  ({int(summary.overall_confidence*5)}/5 features real)')
print(f'Errors: {summary.errors if summary.errors else "none"}')

In [ ]:
# ============================================================================
# Section 17: Layer 3 Validation - Synthetic Patient
#
# SyntheticEnvironmentPatient generates N days of trajectories with KNOWN triggers.
# Validates the EnvironmentAgent end-to-end against three scenarios:
#   Test 1: PM2.5 > 15 -> flare at lag=2d  (positive control, sparse trigger)
#   Test 2: UV > 7 -> flare at lag=1d      (positive control, SLE-like)
#   Test 3: No trigger                     (negative control - MUST be clean)
#
# Calibration notes:
# - Trigger thresholds are at ~p85 of the baseline distributions to fire 10-15%
#   of days. This is conservative (real polluted regions exceed thresholds more
#   frequently). 60-day synthetic window is intentionally tight.
# - BH FDR at 0.10 may suppress weak signals on this conservative test.
#   See Limitation #1 in the notebook header.
# ============================================================================
class SyntheticEnvironmentPatient:
    """Generates daily environmental records with calibrated triggers."""

    TRIGGER_THRESHOLDS = {
        'pm25_ug_m3':            15.0,   # vs N(10, 4)  -> ~10% of days
        'ozone_ppb':             55.0,   # vs N(45, 10) -> ~16% of days
        'uv_index':              7.0,    # vs N(5, 2)   -> ~16% of days
        'barometric_change_kpa': 0.7,    # vs |N(0, 0.4)| -> ~10% of days
        'pollen_index':          6.0,    # vs N(4, 2)   -> ~16% of days
    }

    BASELINE_DISTS = {
        'pm25_ug_m3':            (10.0, 4.0),
        'ozone_ppb':             (45.0, 10.0),
        'uv_index':              (5.0, 2.0),
        'barometric_change_kpa': (0.0, 0.4),   # abs() applied
        'pollen_index':          (4.0, 2.0),
    }

    def __init__(self, triggers=None, flare_lag_days=2,
                 trigger_flare_probability=0.85,
                 baseline_flare_probability=0.10, random_seed=42):
        self.triggers = triggers or []
        self.flare_lag_days = flare_lag_days
        self.trigger_flare_probability = trigger_flare_probability
        self.baseline_flare_probability = baseline_flare_probability
        self.rng = random_mod.Random(random_seed)
        self.np_rng = np.random.RandomState(random_seed)

    def _is_trigger_active(self, features):
        for trig in self.triggers:
            t, v = self.TRIGGER_THRESHOLDS.get(trig), features.get(trig)
            if t is not None and v is not None and v > t:
                return True
        return False

    def generate(self, n_days=60, start_date='2026-03-01'):
        records, scheduled_flares = [], {}
        start = pd.Timestamp(start_date)

        for i in range(n_days):
            date = (start + pd.Timedelta(days=i)).strftime('%Y-%m-%d')
            features = {'date': date}
            for feat, (mean, std) in self.BASELINE_DISTS.items():
                v = self.np_rng.normal(mean, std)
                if feat == 'barometric_change_kpa':
                    v = abs(v)
                if feat == 'uv_index':
                    v = max(0.0, min(11.0, v))
                elif feat == 'pollen_index':
                    v = max(0.0, min(10.0, v))
                else:
                    v = max(0.0, v)
                features[feat] = v

            flare_prob = (self.trigger_flare_probability
                          if self._is_trigger_active(features)
                          else self.baseline_flare_probability)

            if self.rng.random() < flare_prob:
                flare_date = (start + pd.Timedelta(days=i + self.flare_lag_days)).strftime('%Y-%m-%d')
                severity = float(self.np_rng.uniform(1.5, 3.0))
                scheduled_flares[flare_date] = max(scheduled_flares.get(flare_date, 0.0), severity)

            records.append(features)

        return records, scheduled_flares


class _SummaryLike:
    """Lightweight DailyEnvironmentSummary stand-in for the synthetic patient."""
    def __init__(self, d):
        self.date = d['date']
        self.pm25_ug_m3 = d['pm25_ug_m3']
        self.ozone_ppb = d['ozone_ppb']
        self.uv_index = d['uv_index']
        self.barometric_change_kpa = d['barometric_change_kpa']
        self.pollen_index = d['pollen_index']
        self.overall_confidence = 1.0
        self.threshold_alerts = {}


def _run_test(label, patient, expectation):
    records, flares = patient.generate(n_days=60)
    agent = EnvironmentAgent(n_permutations=500)
    for r in records: agent.observe(_SummaryLike(r))
    for d, s in flares.items(): agent.observe_flare(d, s)
    report = agent.analyze()
    print(f'\n{label}')
    print(f'  Expectation: {expectation}')
    print(f'  Observed:     {report.n_days_observed} days, {report.n_flare_events} flare events')
    print(f'  Hypotheses:   {report.n_hypotheses_tested}')
    print(f'  Patterns ({len(report.detected_patterns)} found):')
    for p in report.detected_patterns:
        print(f'    {p.feature:30s} lag={p.lag_days}d  effect={p.effect_size:+.3f}'
              f'  p={p.p_value:.3f}  q={p.q_value:.3f}  [{p.confidence}]')


# ----------------------------------------------------------------------
print('=' * 70)
print('Agent 3 Layer 3 validation - 3 tests')
print('=' * 70)

_run_test(
    'TEST 1: PM2.5 trigger (positive control)',
    SyntheticEnvironmentPatient(triggers=['pm25_ug_m3'], flare_lag_days=2, random_seed=42),
    'pm25_ug_m3 (>p85) at lag=2d (may be suppressed by BH if signal weak)',
)
_run_test(
    'TEST 2: UV trigger (SLE-like patient)',
    SyntheticEnvironmentPatient(triggers=['uv_index'], flare_lag_days=1, random_seed=43),
    'uv_index (>p85) at lag=1d',
)
_run_test(
    'TEST 3: Negative control (MUST be clean)',
    SyntheticEnvironmentPatient(triggers=[], random_seed=44),
    'Zero patterns at [high]; ideally zero patterns at all',
)

print('\n' + '=' * 70)
print('Validation complete. See limitations in notebook header.')
print('=' * 70)